# Controlled changes on the AI-only tracks

The judge ablations of [two_track_figure.ipynb](two_track_figure.ipynb), run on pairs where both
ideas are generated with the same prompt by a stronger (positive) and a weaker (negative) model
of one family: gpt-5.4 vs. gpt-5.1, and claude-opus-4-5 vs. claude-sonnet-4-5.

The label assumes the stronger model's idea is the more novel one. So next to accuracy we
report the **flip rate**: the share of instances whose verdict changes under an ablation,
which needs no label.

Reads the `merge_ai_gpt.yaml` and `merge_ai_claude.yaml` merges.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "output").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.figures.common import FONT_STACK, PDF_RCPARAMS, TwoTrackFigure

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK,
                     **PDF_RCPARAMS})
print(ROOT)

In [ ]:
# (block title, merge dir, track, data file), left to right
TRACKS = [("gpt-5.4 vs. gpt-5.1", "merge_ai_gpt", "pairwise-ai-gpt",
           "data/ai-only/gpt/pairwise.yaml"),
          ("opus-4-5 vs. sonnet-4-5", "merge_ai_claude", "pairwise-ai-claude",
           "data/ai-only/claude/pairwise.yaml")]
ROWS = [
        ("ai_researcher_base", 'P3 · "Which was\njudged novel?"'),
        ("ai_researcher_no_review", 'P4 · "Which is novel?"'),
        ("ai_researcher_comparative", 'P5 · "Which was judged\n more novel?"')]
MODELS = ["claude-sonnet-4-5", "claude-opus-4-5", "claude-opus-4-6",
          "gpt-5.1", "gpt-5.2", "gpt-5.4"]

FIG = TwoTrackFigure(root=ROOT, figure="ai-only",
                     tracks=[(title, merge, "") for title, merge, *_ in TRACKS],
                     rows=ROWS, models=MODELS,
                     metrics={"pairwise": "pairwise_accuracy_strict"},
                     row_groups=[("Judge prompt", [name for name, _ in ROWS])],
                     row_axis_label="Change", baseline_label="Reference")

## Accuracy

In [ ]:
print(FIG.table("pairwise").round(1).to_string(index=False))
png = FIG.draw("pairwise")
print(png.relative_to(ROOT), "(+ .pdf)")
display(Image(filename=str(png)))

## Flip rate

% of instances whose verdict (idea 0, idea 1 or tie) differs from the same judge's
verdict under `current`. Averaged over runs when an ablation has several.

In [ ]:
TYPES = ["✓→✗", "✗→✓", "✓→tie", "✗→tie", "tie→✓", "tie→✗"]


def verdicts(merge, track, ablation, model):
    """One {instance id: verdict} dict per run, from the newest artifact dir with scores."""
    artifacts = ROOT / "output/ablation_sweeps" / merge / "artifacts"
    files = sorted(artifacts.glob(f"{track}_{ablation}-{model}_*/run_*/scores.json"))
    newest = [f for f in files if f.parents[1] == files[-1].parents[1]]
    return [{k: v["comparisons"][0]["winner"] for k, v in json.loads(f.read_text()).items()}
            for f in newest]


def breakdown(merge, track, data, ref, rows):
    """Instances per kind of flip from `ref` to each of `rows`, per judge."""
    gold = {str(k): v["expected_winners"][0]
            for k, v in yaml.safe_load((ROOT / data).read_text()).items()}

    def mark(i, verdict):
        return "tie" if verdict == 2 else "✓" if verdict == gold[i] else "✗"

    recs = []
    for model in MODELS:
        base = verdicts(merge, track, ref, model)[0]
        for name, label in rows:
            runs = verdicts(merge, track, name, model)
            for run in runs:
                for i in base.keys() & run.keys():
                    if base[i] != run[i]:
                        recs.append({"ablation": label, "judge": model, "n": 1 / len(runs),
                                     "flip": f"{mark(i, base[i])}→{mark(i, run[i])}"})
    table = (pd.DataFrame(recs)
             .pivot_table(index=["ablation", "judge"], columns="flip", values="n", aggfunc="sum")
             .reindex(index=pd.MultiIndex.from_product([[label for _, label in rows], MODELS]),
                      columns=TYPES)
             .fillna(0))
    table["total"] = table.sum(axis=1)
    table["%"] = 100 * table["total"] / len(gold)
    return table


VS_CURRENT = {title: breakdown(*spec, "current", ROWS) for title, *spec in TRACKS}
FLIPS = pd.concat({title: t["%"].unstack().reindex(index=[label for _, label in ROWS],
                                                    columns=MODELS)
                   for title, t in VS_CURRENT.items()})
print(FLIPS.round(1).to_string())

### Flip breakdown

Number of instances per kind of flip, reference → ablation: ✓ correct, ✗ wrong, tie.
`%` is the total over all instances. Averaged over runs when an ablation has several.
Shown against `current`, and against P3 for the other minimal prompts.

In [ ]:
OTHERS = [row for row in ROWS if row[0] != "ai_researcher_base"]
for title, *spec in TRACKS:
    print(f"{title} · reference: current")
    print(VS_CURRENT[title].round(1).to_string(), "\n")
    print(f"{title} · reference: P3")
    print(breakdown(*spec, "ai_researcher_base", OTHERS).round(1).to_string(), "\n")